In [10]:
all_datasets = ['real_data/data_batteries_ecfp_descriptor', 'synthetic_data/qm9_simple_linear6', 'synthetic_data/qm9_piecewise_linear_6','synthetic_data/qm9_nonlinear_6']
dataset_names = ['Batteries', 'QM9 Simple Linear', 'QM9 Piecewise Linear', 'QM9 Nonlinear']
metrics = {}
metrics_top10 = {}

import pandas as pd
import pickle

for i, dataset in enumerate(all_datasets):
    with open(f'../results/{dataset}/explanations/analysis/metrics_results.pickle', 'rb') as f:
        results = pickle.load(f)
    metrics[dataset_names[i]] = results

metrics

{'Batteries': {'lime': {'pgi_mean': np.float64(0.40335898273904036),
   'pgu_mean': np.float64(0.15221911540208974),
   'pgi_std': np.float64(0.08465451424520036),
   'pgu_std': np.float64(0.052931360998383054),
   'pgi_mean_10': np.float64(0.2698437375833648),
   'pgu_mean_10': np.float64(0.030804127609349957),
   'pgi_std_10': np.float64(0.051327471905348414),
   'pgu_std_10': np.float64(0.020606336311041636),
   'pgi_org_mean': np.float64(93.15522141201139),
   'pgu_org_mean': np.float64(33.99381005944482),
   'pgi_org_std': np.float64(17.595259375776287),
   'pgu_org_std': np.float64(8.020631717956793),
   'pgi_org_mean_10': np.float64(63.732756460338045),
   'pgu_org_mean_10': np.float64(6.740289293921767),
   'pgi_org_std_10': np.float64(16.630849891442686),
   'pgu_org_std_10': np.float64(4.141691055730131)},
  'shap': {'pgi_mean': np.float64(0.40716324934868736),
   'pgu_mean': np.float64(0.12372224917585768),
   'pgi_std': np.float64(0.10116486705267498),
   'pgu_std': np.floa

In [13]:
metrics_dataframe_normalized = {'method': [], 'dataset': [], 'metric': [], 'value': []}
metrics_dataframe = {'method': [], 'dataset': [], 'metric': [], 'value': []}
metrics_top10_dataframe_normalized = {'method': [], 'dataset': [], 'metric': [], 'value': []}
metrics_top10_dataframe = {'method': [], 'dataset': [], 'metric': [], 'value': []}

for dataset in dataset_names:
    for method, method_results in metrics[dataset].items():

        metrics_dataframe['method'].append(method)
        metrics_dataframe['dataset'].append(dataset)
        metrics_dataframe['method'].append(method)
        metrics_dataframe['dataset'].append(dataset)
        metrics_dataframe['metric'].append('pgi')
        metrics_dataframe['value'].append(method_results['pgi_org_mean'])
        metrics_dataframe['metric'].append('pgu')
        metrics_dataframe['value'].append(method_results['pgu_org_mean'])


        metrics_dataframe_normalized['method'].append(method)
        metrics_dataframe_normalized['dataset'].append(dataset)
        metrics_dataframe_normalized['method'].append(method)
        metrics_dataframe_normalized['dataset'].append(dataset)
        metrics_dataframe_normalized['metric'].append('pgi')
        metrics_dataframe_normalized['value'].append(method_results['pgi_mean'])
        metrics_dataframe_normalized['metric'].append('pgu')
        metrics_dataframe_normalized['value'].append(method_results['pgu_mean'])

        metrics_top10_dataframe['method'].append(method)
        metrics_top10_dataframe['dataset'].append(dataset)
        metrics_top10_dataframe['method'].append(method)
        metrics_top10_dataframe['dataset'].append(dataset)
        metrics_top10_dataframe['metric'].append('pgi')
        metrics_top10_dataframe['value'].append(method_results['pgi_org_mean_10'])
        metrics_top10_dataframe['metric'].append('pgu')
        metrics_top10_dataframe['value'].append(method_results['pgu_org_mean_10'])

        metrics_top10_dataframe_normalized['method'].append(method)
        metrics_top10_dataframe_normalized['dataset'].append(dataset)
        metrics_top10_dataframe_normalized['method'].append(method)
        metrics_top10_dataframe_normalized['dataset'].append(dataset)
        metrics_top10_dataframe_normalized['metric'].append('pgi')
        metrics_top10_dataframe_normalized['value'].append(method_results['pgi_mean_10'])
        metrics_top10_dataframe_normalized['metric'].append('pgu')
        metrics_top10_dataframe_normalized['value'].append(method_results['pgu_mean_10'])


In [14]:
metrics_dataframe = pd.DataFrame(metrics_dataframe)
metrics_dataframe_normalized = pd.DataFrame(metrics_dataframe_normalized)
metrics_top10_dataframe = pd.DataFrame(metrics_top10_dataframe)
metrics_top10_dataframe_normalized = pd.DataFrame(metrics_top10_dataframe_normalized)

In [15]:
metrics_dataframe

,method,dataset,metric,value
0,lime,Batteries,pgi,93.155221
1,lime,Batteries,pgu,33.993810
2,shap,Batteries,pgi,93.912616
3,shap,Batteries,pgu,27.443700
4,shapiq1,Batteries,pgi,91.419256
5,shapiq1,Batteries,pgu,38.440561
6,shapiq2,Batteries,pgi,92.251775
7,shapiq2,Batteries,pgu,57.446668
8,meg,Batteries,pgi,79.598912
9,meg,Batteries,pgu,45.381404


In [18]:
def reshape_results_to_wide_format(df: pd.DataFrame) -> pd.DataFrame:
    """
    Transforms a long-format DataFrame of results into a wide-format DataFrame.

    The resulting DataFrame will have 'method' as the index, and a single column
    for each combination of 'dataset' and 'metric'.

    Args:
        df (pd.DataFrame): A DataFrame with columns ['method', 'dataset', 'metric', 'value'].

    Returns:
        pd.DataFrame: The reshaped, wide-format DataFrame.
    """
    # --- Step 1: Use pivot_table to reshape the data ---
    # - 'index' specifies the rows of the new DataFrame.
    # - 'columns' specifies the columns to "unstack" to create the new columns.
    # - 'values' specifies what will fill the cells.
    pivoted_df = df.pivot_table(
        index='method',
        columns=['dataset', 'metric'],
        values='value'
    )

    # --- Step 2: Flatten the MultiIndex columns ---
    # The columns are now tuples like ('DS1', 'PGI'). We join them into a single string.
    pivoted_df.columns = [f"{col[0]}_{col[1]}" for col in pivoted_df.columns]

    # --- Optional Step 3: Reset the index if you want 'method' as a column ---
    # pivoted_df = pivoted_df.reset_index()

    return pivoted_df

In [19]:
metrics_dataframe = reshape_results_to_wide_format(metrics_dataframe)
metrics_dataframe_normalized = reshape_results_to_wide_format(metrics_dataframe_normalized)
metrics_top10_dataframe = reshape_results_to_wide_format(metrics_top10_dataframe)
metrics_top10_dataframe_normalized = reshape_results_to_wide_format(metrics_top10_dataframe_normalized)

In [20]:
metrics_dataframe

,Batteries_pgi,Batteries_pgu,QM9 Nonlinear_pgi,QM9 Nonlinear_pgu,QM9 Piecewise Linear_pgi,QM9 Piecewise Linear_pgu,QM9 Simple Linear_pgi,QM9 Simple Linear_pgu
method,,,,,,,,
aggregated,93.435320,31.596216,15.780227,1.833475,14.460313,1.894665,15.649869,1.818925
lime,93.155221,33.993810,15.881058,1.756983,14.485523,2.030060,15.799401,1.790133
meg,79.598912,45.381404,15.357054,3.764146,14.826237,3.261669,15.402258,2.533653
mmace,81.576001,42.041934,15.492937,2.949218,14.815452,3.414720,15.937290,2.476065
shap,93.912616,27.443700,15.870401,1.545215,14.410748,1.570404,15.680972,1.631089
shapiq1,91.419256,38.440561,15.804157,1.800169,14.298902,2.115520,15.590307,1.908390
shapiq2,92.251775,57.446668,15.781305,2.204978,14.340403,3.779812,15.708857,2.893422


In [21]:
metrics_top10_dataframe

,Batteries_pgi,Batteries_pgu,QM9 Nonlinear_pgi,QM9 Nonlinear_pgu,QM9 Piecewise Linear_pgi,QM9 Piecewise Linear_pgu,QM9 Simple Linear_pgi,QM9 Simple Linear_pgu
method,,,,,,,,
aggregated,64.803383,1.249266,15.246676,0.487294,14.216013,0.291456,14.352307,0.532782
lime,63.732756,6.740289,15.720445,0.613647,14.544019,0.487789,14.613808,0.686972
meg,28.964878,8.410473,14.862631,1.431877,15.315145,1.112357,14.395078,0.934774
mmace,49.283540,8.463395,14.805523,0.953231,14.713589,1.194808,14.749151,0.896116
shap,73.941514,0.864033,15.823477,0.380945,14.359572,0.202271,14.511684,0.400915
shapiq1,64.805124,2.248775,15.284608,0.432283,13.775585,0.218256,13.969889,0.471951
shapiq2,58.015387,9.040767,15.446949,0.736663,13.835141,0.705369,13.981727,0.921566


In [22]:
metrics_dataframe.to_latex()

'\\begin{tabular}{lrrrrrrrr}\n\\toprule\n & Batteries_pgi & Batteries_pgu & QM9 Nonlinear_pgi & QM9 Nonlinear_pgu & QM9 Piecewise Linear_pgi & QM9 Piecewise Linear_pgu & QM9 Simple Linear_pgi & QM9 Simple Linear_pgu \\\\\nmethod &  &  &  &  &  &  &  &  \\\\\n\\midrule\naggregated & 93.435320 & 31.596216 & 15.780227 & 1.833475 & 14.460313 & 1.894665 & 15.649869 & 1.818925 \\\\\nlime & 93.155221 & 33.993810 & 15.881058 & 1.756983 & 14.485523 & 2.030060 & 15.799401 & 1.790133 \\\\\nmeg & 79.598912 & 45.381404 & 15.357054 & 3.764146 & 14.826237 & 3.261669 & 15.402258 & 2.533653 \\\\\nmmace & 81.576001 & 42.041934 & 15.492937 & 2.949218 & 14.815452 & 3.414720 & 15.937290 & 2.476065 \\\\\nshap & 93.912616 & 27.443700 & 15.870401 & 1.545215 & 14.410748 & 1.570404 & 15.680972 & 1.631089 \\\\\nshapiq1 & 91.419256 & 38.440561 & 15.804157 & 1.800169 & 14.298902 & 2.115520 & 15.590307 & 1.908390 \\\\\nshapiq2 & 92.251775 & 57.446668 & 15.781305 & 2.204978 & 14.340403 & 3.779812 & 15.708857 & 2.893

In [23]:
metrics_top10_dataframe.to_latex()

'\\begin{tabular}{lrrrrrrrr}\n\\toprule\n & Batteries_pgi & Batteries_pgu & QM9 Nonlinear_pgi & QM9 Nonlinear_pgu & QM9 Piecewise Linear_pgi & QM9 Piecewise Linear_pgu & QM9 Simple Linear_pgi & QM9 Simple Linear_pgu \\\\\nmethod &  &  &  &  &  &  &  &  \\\\\n\\midrule\naggregated & 64.803383 & 1.249266 & 15.246676 & 0.487294 & 14.216013 & 0.291456 & 14.352307 & 0.532782 \\\\\nlime & 63.732756 & 6.740289 & 15.720445 & 0.613647 & 14.544019 & 0.487789 & 14.613808 & 0.686972 \\\\\nmeg & 28.964878 & 8.410473 & 14.862631 & 1.431877 & 15.315145 & 1.112357 & 14.395078 & 0.934774 \\\\\nmmace & 49.283540 & 8.463395 & 14.805523 & 0.953231 & 14.713589 & 1.194808 & 14.749151 & 0.896116 \\\\\nshap & 73.941514 & 0.864033 & 15.823477 & 0.380945 & 14.359572 & 0.202271 & 14.511684 & 0.400915 \\\\\nshapiq1 & 64.805124 & 2.248775 & 15.284608 & 0.432283 & 13.775585 & 0.218256 & 13.969889 & 0.471951 \\\\\nshapiq2 & 58.015387 & 9.040767 & 15.446949 & 0.736663 & 13.835141 & 0.705369 & 13.981727 & 0.921566 \\\